# Reinforcement Learning for Large Language Models (RLHF)



# 1. Mapping Reinforcement Learning to Large Language Models

In classical Reinforcement Learning we have:

| RL Component | Meaning |
|---|---|
| Agent | Entity making decisions |
| State | Current environment |
| Action | Decision taken |
| Reward | Feedback signal |
| Policy | Strategy used by agent |

To apply this framework to **LLMs**, we reinterpret these components.


## RL → LLM Mapping

| RL Component | LLM Equivalent |
|---|---|
| **Agent** | The Large Language Model |
| **State (s)** | Prompt + conversation history |
| **Action (a)** | Token generated by the model |
| **Reward (R)** | Human feedback / Reward model |
| **Policy (π)** | Probability distribution over next tokens |

Example:

Prompt:
```
Explain gravity.
```

State:
```
Prompt + tokens already generated
```

Action:
```
Next predicted token
```


## Practical Example: Token Generation as Actions

A language model predicts the probability distribution over tokens.

Example:

```
Input: "The sky is"
```
Possible next tokens:

| Token | Probability |
|---|---|
| blue | 0.65 |
| dark | 0.15 |
| falling | 0.02 |

The **policy** of the model is the probability distribution used to select these tokens.


# 2. Sequence Generation as an RL Problem

Text generation is a **sequential decision process**.

At each step:

State:
```
Prompt + tokens generated so far
```

Action:
```
Next token predicted by the model
```

The final output is a **sequence of actions**.


## 🔹 The RL Policy

The policy defines how the agent chooses an action given a state.

$$
\pi(a|s)
$$

Where:

- \( s \) = current state  
- \( a \) = action  
- \( \pi(a|s) \) = probability of choosing action \( a \) in state \( s \)  

---

## 🔹 Mapping to LLMs

In Large Language Models:

- **State** = prompt + generated tokens  
- **Action** = next token  
- **Policy** = softmax distribution over vocabulary  

---

## 🧠 Intuition

```
Given a sentence so far → model predicts probability of next word
```

Example:

Prompt:
```
"The cat sat on the"
```

Policy Output:
- mat → 0.6  
- floor → 0.3  
- table → 0.1  

👉 Model samples based on this probability distribution

## Deterministic vs Stochastic Policy

### Deterministic Policy

Always produces the same output for the same state.

Example:
```
if health < 20 -> heal
```

Formula:

π(s) = a

### Stochastic Policy

Selects actions probabilistically.

Example:

| Action | Probability |
|---|---|
Attack | 0.6 |
Defend | 0.3 |
Run | 0.1 |

Deep RL and LLMs **use stochastic policies**.


## Sequence Level Reward

In LLM training, reward is typically provided **after the full response is generated**.

Example:

Prompt:
```
Explain gravity
```

Generated Response:

```
Gravity is a force that attracts objects with mass.
```

Only **after the entire response** is produced do we assign a reward score.


# 3. Reward Modeling

The model needs a mechanism to determine which responses are good.

Instead of humans evaluating every response during RL training, we train a **Reward Model**.


## Human Preference Collection

Humans compare multiple outputs for the same prompt.

Example dataset:

Prompt:
```
Explain gravity
```

Output A:
```
Gravity pulls objects toward each other.
```

Output B:
```
Gravity is when things fall sometimes.
```

Human chooses **A**.



## Bradley–Terry Model

Used to convert human rankings into reward scores.

Probability that response \(i\) is preferred over \(j\):

$$
P(i > j) = \frac{e^{r_i}}{e^{r_i} + e^{r_j}}
$$
Where:

- \(r_i\) = reward score predicted by model
- \(r_j\) = reward score for another response


In [ ]:
# Simple reward model example

import torch
import torch.nn as nn

class RewardModel(nn.Module):

    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

# simulate embeddings for responses
response_a = torch.randn(1,10)
response_b = torch.randn(1,10)

model = RewardModel(10)

score_a = model(response_a)
score_b = model(response_b)

print("Score A:", score_a.item())
print("Score B:", score_b.item())


Score A: 0.06395773589611053
Score B: 0.056897059082984924


In [ ]:
# Bradley-Terry probability

import torch.nn.functional as F

prob_a_preferred = torch.exp(score_a) / (torch.exp(score_a) + torch.exp(score_b))

print("Probability A preferred:", prob_a_preferred.item())


Probability A preferred: 0.5017651319503784


# 4. RLHF Pipeline

Reinforcement Learning from Human Feedback consists of multiple training stages.


## Stage 1 — Pretraining

The base LLM is trained using standard **next-token prediction** on massive datasets.


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen1.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

# Add this line to define a padding token
tokenizer.pad_token = tokenizer.eos_token

print("Base LLM Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base LLM Loaded


In [2]:
prompt = "Summarize this article about climate change:"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = base_model.generate(
    **inputs,
    max_length=50,
    do_sample=True, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.2
)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize this article about climate change: "Global warming is the rate of increase in average global temperatures over time. The primary drivers of global temperature rise are greenhouse gases (GHGs). GHG emissions, primarily from human activities such as burning fossil fuels and deforestation, trap heat inside Earth's atmosphere through processes called Greenhouse Gas Emission Concentrations or GEGC."<|endoftext|>


## Stage 2 — Supervised Fine Tuning (SFT)

The model is trained on **instruction datasets**.

Example:

Prompt:
```
Summarize this article
```

Target Output:
```
Short summary written by humans
```


In [3]:
sft_dataset = [
    {
        "prompt": "Summarize this article",
        "response": "Climate change is causing rising temperatures and extreme weather."
    },
    {
        "prompt": "Explain gravity",
        "response": "Gravity is the force that attracts objects with mass toward each other."
    }
]

In [4]:
import torch

def tokenize(example):

    text = example["prompt"] + " " + example["response"]

    return tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_tensors="pt"
    )

In [5]:
from torch.optim import AdamW

optimizer = AdamW(base_model.parameters(), lr=5e-5)

base_model.train()

for epoch in range(2):

    for example in sft_dataset:

        inputs = tokenize(example)

        labels = inputs["input_ids"].clone()
        labels[inputs["attention_mask"] == 0] = -100

        outputs = base_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Epoch completed:", epoch)

Epoch completed: 0
Epoch completed: 1


## Stage 3 — Collect Human Preferences

Multiple responses are generated.

Humans rank them.

Example:

| Response | Rank |
|---|---|
Response A | 1 |
Response B | 2 |
Response C | 3 |


In [6]:
prompt = "Explain gravity"

inputs = tokenizer(prompt, return_tensors="pt")

output1 = base_model.generate(**inputs, max_length=40, do_sample=True, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.2)
output2 = base_model.generate(**inputs, max_length=40, do_sample=True, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.2)

resp1 = tokenizer.decode(output1[0])
resp2 = tokenizer.decode(output2[0])

print("Response A:\n", resp1)
print("\nResponse B:\n", resp2)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response A:
 Explain gravity Gravity is the force that attracts objects with mass toward each other.<|endoftext|>

Response B:
 Explain gravity Gravity is the force that attracts objects with mass toward each other. The greater an object has weight, the more it will be attracted by another object with less weight. A large amount of gravity can attract objects moving towards one another.<|endoftext|>


In [7]:
preference_dataset = [

{
    "prompt": prompt,
    "chosen": resp1,
    "rejected": resp2
}

]

## Stage 4 — Train Reward Model

The reward model learns to predict which responses humans prefer.


In [8]:
import torch.nn as nn

class RewardModel(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        score = self.linear(hidden_states)
        return score


In [9]:
reward_model = RewardModel(hidden_size=base_model.config.hidden_size).to(base_model.dtype).to(base_model.device)

In [10]:
import torch.nn.functional as F

optimizer = AdamW(reward_model.parameters(), lr=1e-4)

for example in preference_dataset:

    chosen = tokenizer(example["chosen"], return_tensors="pt", truncation=True, padding=True)
    rejected = tokenizer(example["rejected"], return_tensors="pt", truncation=True, padding=True)

    # FIXED: changed 'transformer' to 'model'
    chosen_outputs = base_model.model(**chosen)
    rejected_outputs = base_model.model(**rejected)

    # Grab the last token of the sequence (for batch size 1)
    chosen_embed = chosen_outputs.last_hidden_state[:, -1, :]
    rejected_embed = rejected_outputs.last_hidden_state[:, -1, :]

    chosen_score = reward_model(chosen_embed)
    rejected_score = reward_model(rejected_embed)

    # Bradley Terry loss
    loss = -torch.log(torch.sigmoid(chosen_score - rejected_score)).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print("Reward Model Loss:", loss.item())


Reward Model Loss: 2.1875


## Stage 5 — RL Optimization

The LLM is optimized using Reinforcement Learning so it produces high-reward responses.


In [11]:
prompt = "Explain gravity simply"

inputs = tokenizer(prompt, return_tensors="pt")

output = base_model.generate(
    **inputs,
    max_length=50,
    do_sample=True, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.2
)

response = tokenizer.decode(output[0])
print(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain gravity simply Gravity is the force that attracts objects with mass toward each other.<|endoftext|>


### KL Penalty

To prevent the model from drifting too far from the base model, we introduce a **KL divergence penalty**.

$$
\text{Total Reward} = Reward_{model} - \beta \cdot KL(\pi \parallel \pi_{ref})
$$

Where:

- $\pi$ = current policy  
- $\pi_{ref}$ = reference/original model policy  
- $\beta$ = KL penalty coefficient that controls how strongly we keep the new policy close to the reference policy

# 5. PPO for LLM Training

Proximal Policy Optimization is the most common RL algorithm used in RLHF.


## Why PPO?

Vanilla policy gradients are unstable.

PPO stabilizes training using **clipped objectives**.


# PPO Clipped Objective

$$
L^{CLIP}(\theta) = \mathbb{E} \Big[
\min \big( r(\theta) A,\; \text{clip}(r(\theta), 1 - \epsilon,\; 1 + \epsilon) \cdot A \big)
\Big]
$$

Where:
- $r(\theta) = \frac{\pi_{\theta}(a|s)}{\pi_{\text{old}}(a|s)}$ → Probability ratio
- $A$ → Advantage estimate
- $\epsilon$ → Clipping parameter (e.g., 0.2)

---

## 🔹 1. Probability Ratio $r(\theta)$

$$
r(\theta) = \frac{\pi_{\theta}(a|s)}{\pi_{\text{old}}(a|s)}
$$

> **How much did the new policy change compared to the old policy?**

| Value | Meaning |
|-------|---------|
| $r = 1$ | No change — policy stayed the same |
| $r > 1$ | Probability increased — action is more likely now |
| $r < 1$ | Probability decreased — action is less likely now |

---

## 🔹 2. Advantage $A$

$$
A(s, a) = Q(s, a) - V(s)
$$

> **Was this action better or worse than expected?**

It is not asking *"was this good?"*
It is asking *"was this **better than what I expected?**"*



### In LLM Terms

```
Prompt   →  "Explain gravity"
Response →  Model generates an answer
Reward   →  1.5
Expected →  1.0  (baseline)
─────────────────────────────────────
Advantage = 1.5 - 1.0 = +0.5 ✅
→ Better than expected → increase its probability
```

### ❓ Why Not Just Use Raw Reward?

```
Response 1 → reward 8  (easy prompt)
Response 2 → reward 5  (very hard prompt)
```

Raw reward says Response 1 is better.
But for that hard prompt, **5 might be outstanding.**

```
Response 1 → 8 - 8 = 0    just as expected
Response 2 → 5 - 2 = +3   much better than expected ✅
```

> Advantage removes prompt difficulty and captures **relative quality.**

### Effect on Policy

| Value | Meaning | Effect |
|-------|---------|--------|
| $A > 0$ | Better than expected | Increase probability |
| $A < 0$ | Worse than expected | Decrease probability |

---

## 🔹 3. How PPO Calculates Advantage — GAE

PPO trains a separate **value network (critic)** to predict the baseline $V(s)$.

```
Step 1: Value network sees current state
        → predicts expected reward = 1.0

Step 2: Actual reward = 1.5

Step 3: Advantage = 1.5 - 1.0 = +0.5
```

PPO computes advantage **per token** using GAE (Generalized Advantage Estimation):

$$
A_t = \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + \cdots
$$

Where each TD error $\delta_t$ is:

$$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

```
Tokens generated: ["Gravity", "is", "a", "force"]
                    t=0       t=1   t=2   t=3

Only t=3 gets the reward (end of response)
GAE rolls backwards from t=3 → t=0
giving each token its own advantage estimate
```

> ⚠️ This requires a **value network** — a second neural network.
> This is why PPO needs 4 models total.

---

## 🔹 4. Reward Model vs Value Network

> One of the most confusing parts of PPO — they both output numbers, but they answer completely different questions.

| | Reward Model | Value Network |
|---|---|---|
| **Question** | Was this response good? | How good is this state right now? |
| **Input** | Full completed response | Current token / state |
| **Output** | Final scalar score | Expected future reward |
| **When it runs** | Once — at end of response | At every single token |
| **Trained on** | Human preferences | TD errors during PPO |
| **Changes during PPO?** | ❌ Frozen | ✅ Trains alongside policy |

### 🍕 Pizza Analogy

```
Reward Model  →  Food critic
                 Reads the full review after eating
                 Gives a final score → 8/10

Value Network →  Your own gut feeling
                 At each bite predicts "this will probably be 7/10"
                 Updates as you keep eating
```

### In LLM Terms

```
Prompt: "Explain gravity"

Model generates token by token:
"Gravity" → "is" → "a" → "force" → "that" → "pulls"
```

**Reward Model** — runs once at the end:
```
Reads full response: "Gravity is a force that pulls"
→ outputs score = 1.5
Frozen. Done.
```

**Value Network** — runs at every token:
```
After "Gravity"            → V(s) = 0.9
After "Gravity is"         → V(s) = 1.0
After "Gravity is a"       → V(s) = 1.1
After "Gravity is a force" → V(s) = 1.4

Trainable. Updates every step.
```

### How They Work Together

```
Step 1 — Reward Model scores full response
         "Gravity is a force that pulls" → 1.5

Step 2 — Value Network predicted at last token
         V(s) = 1.3

Step 3 — TD error at last token
         δ = 1.5 - 1.3 = +0.2

Step 4 — GAE rolls backwards
         distributes credit to every token

Step 5 — PPO updates policy using per-token advantage
```

> Reward model provides the **signal.**
> Value network converts that signal into **per-token advantage.**
> Without the value network, you only know the response was good —
> but not **which tokens** made it good.

---

## 🔹 5. First Term — Basic Policy Gradient

$$
r(\theta) \cdot A
$$

> **Increase probability of good actions. Decrease probability of bad actions.**

```
Good action (A > 0) → r·A is positive → push probability up
Bad  action (A < 0) → r·A is negative → push probability down
```

⚠️ **Problem** — if $r$ is very large, the update becomes too big.
The policy can collapse or diverge. This is why clipping is needed.

---

## 🔹 6. Clipping Function

$$
\text{clip}(r(\theta),\ 1-\epsilon,\ 1+\epsilon)
$$

> **If the policy changed too much — limit it.**

With $\epsilon = 0.2$, the allowed range is **[0.8, 1.2]**:

| $r$ value | After clip | What happened |
|-----------|------------|---------------|
| 1.1 | 1.1 | Inside range — no change |
| 1.5 | 1.2 | Too large — clipped down |
| 0.6 | 0.8 | Too small — clipped up |

---

## 🔹 7. Second Term — Clipped Objective

$$
\text{clip}(r(\theta),\ 1-\epsilon,\ 1+\epsilon) \cdot A
$$

> This is the **safe version** of the update — ratio is bounded, so updates stay controlled.

---

## 🔥 8. Why `min()` is the Core of PPO ⭐

$$
\min\left(r(\theta) \cdot A,\ \text{clip}(r(\theta), 1-\epsilon, 1+\epsilon) \cdot A\right)
$$

PPO always picks the **more conservative** of the two terms.
This means it **never over-rewards a lucky update.**

---

### Case 1: Good Action $(A > 0)$

We want to increase probability — but not too much.

```
r = 1.5  (policy changed a lot)
A = 1.0  (good action)

Unclipped = 1.5 × 1.0 = 1.5
Clipped   = 1.2 × 1.0 = 1.2

min(1.5, 1.2) = 1.2  ✅
```

> PPO picks **1.2** — increases probability, but not as aggressively as the raw gradient wanted.

---

### Case 2: Bad Action $(A < 0)$

We want to decrease probability — but not too aggressively.

```
r = 0.5   (policy changed a lot in opposite direction)
A = -1.0  (bad action)

Unclipped = 0.5 × -1.0 = -0.5
Clipped   = 0.8 × -1.0 = -0.8

min(-0.5, -0.8) = -0.8  ✅
```

> PPO picks **-0.8** — decreases probability, but prevents the policy from over-correcting.

---

## 💡 One Line Summary

```
Reward Model  → "Was the full response good?"     (final judge, frozen)
Value Network → "How good is where we are now?"   (live predictor, trainable)

Unclipped term → what the gradient wants to do
Clipped term   → the safe bounded version

min() always picks the safer one
→ PPO never over-updates in either direction
```

> The clipping does not make PPO conservative overall —
> it just **prevents single large updates from destabilizing training.**

## Conceptual PPO Training Loop

1. Generate responses using current LLM
2. Score responses using reward model
3. Compute advantage
4. Update model using PPO


# GRPO Clipped Objective

$$
L^{GRPO}(\theta) = \mathbb{E} \left[ \frac{1}{G} \sum_{i=1}^{G} \min \left( r_i(\theta) \hat{A}_i,\ \text{clip}(r_i(\theta), 1-\epsilon, 1+\epsilon) \cdot \hat{A}_i \right) - \beta \cdot \mathbb{D}_{KL}\left[\pi_\theta \| \pi_{ref}\right] \right]
$$

Where:
- $G$ → Group size (number of responses sampled per prompt)
- $r_i(\theta) = \frac{\pi_{\theta}(o_i|q)}{\pi_{old}(o_i|q)}$ → Probability ratio for response $i$
- $\hat{A}_i$ → Group-relative advantage for response $i$
- $\epsilon$ → Clipping parameter (e.g., 0.2)
- $\beta$ → KL penalty coefficient

---

## 🔹 1. Probability Ratio $r_i(\theta)$

$$
r_i(\theta) = \frac{\pi_{\theta}(o_i|q)}{\pi_{old}(o_i|q)}
$$

> **How much did the new policy change compared to the old policy for response $i$?**

| Value | Meaning |
|-------|---------|
| $r_i = 1$ | No change — policy stayed the same |
| $r_i > 1$ | Probability increased — response is more likely now |
| $r_i < 1$ | Probability decreased — response is less likely now |

Same as PPO — only difference is it is computed **per response** not per token.

---

## 🔹 2. Group-Relative Advantage $\hat{A}_i$

$$
\hat{A}_i = \frac{r_i - \text{mean}(\{r_1, r_2, ..., r_G\})}{\text{std}(\{r_1, r_2, ..., r_G\})}
$$

> **Was this response better or worse than the group average?**

It is not asking *"was this response good?"*
It is asking *"was this **better than the other responses for the same prompt?**"*

### 🍕 Simple Intuition

Four students answer the same exam question.
Their scores are: 90, 50, 70, 30

```
Average = 60

Student 1 → 90 - 60 = +30  ✅ above average
Student 2 → 50 - 60 = -10  ❌ below average
Student 3 → 70 - 60 = +10  ✅ above average
Student 4 → 30 - 60 = -30  ❌ below average
```

> The group itself is the baseline — no external judge needed.

### In LLM Terms

```
Prompt: "Explain gravity simply"

GRPO samples G=4 responses:

Response 1 → "Gravity is a force that pulls"   → reward 1.5
Response 2 → "Gravity means things fall"        → reward 0.5
Response 3 → "Newton discovered gravity"        → reward 1.0
Response 4 → "Gravity is mass attraction"       → reward 0.0

Group mean = 0.75
Group std  = 0.56
```

| Response | Reward | Mean | Std | $\hat{A}_i$ | |
|----------|--------|------|-----|-------------|---|
| 1 | 1.5 | 0.75 | 0.56 | **+1.34** | ✅ best |
| 2 | 0.5 | 0.75 | 0.56 | **−0.45** | ❌ below avg |
| 3 | 1.0 | 0.75 | 0.56 | **+0.45** | ✅ above avg |
| 4 | 0.0 | 0.75 | 0.56 | **−1.34** | ❌ worst |

### Effect on Policy

| Value | Meaning | Effect |
|-------|---------|--------|
| $\hat{A}_i > 0$ | Better than group average | Increase probability |
| $\hat{A}_i < 0$ | Worse than group average | Decrease probability |

---

## 🔹 3. How GRPO Calculates Advantage — No Critic Needed

> This is where GRPO fundamentally differs from PPO.

PPO trains a **value network** to predict the baseline:
```
Value network → V(s) = 1.3  (neural network prediction)
Advantage     = reward - V(s)
```

GRPO uses **group statistics** as the baseline:
```
Group rewards → [1.5, 0.5, 1.0, 0.0]
Mean          = 0.75  ← this IS the baseline
Advantage     = (reward - mean) / std
```

### Step by Step

```
Step 1 — Sample G=4 responses from same prompt
Step 2 — Score each with reward function
         [1.5, 0.5, 1.0, 0.0]
Step 3 — Compute group mean and std
         mean = 0.75,  std = 0.56
Step 4 — Normalize each reward
         A1 = (1.5 - 0.75) / 0.56 = +1.34
         A2 = (0.5 - 0.75) / 0.56 = -0.45
         A3 = (1.0 - 0.75) / 0.56 = +0.45
         A4 = (0.0 - 0.75) / 0.56 = -1.34
```

> No neural network. No training. Just statistics.

### ❓ Why Not Just Use Raw Reward?

```
Prompt 1 (easy): "What is 2+2?"
  Response → reward 0.9   (easy to get high reward)

Prompt 2 (hard): "Explain quantum entanglement"
  Response → reward 0.6   (hard to get high reward)
```

Raw reward says Prompt 1 response is better.
But within its group, the Prompt 2 response might be outstanding.

Group normalization fixes this:
```
Prompt 1 group avg = 0.85 → A = 0.9 - 0.85 = +0.05  (barely above avg)
Prompt 2 group avg = 0.3  → A = 0.6 - 0.30 = +0.30  (well above avg) ✅
```

> Advantage captures **relative quality within the group** — not absolute reward.

---

## 🔹 4. First Term — Basic Policy Gradient

$$
r_i(\theta) \cdot \hat{A}_i
$$

> **Increase probability of above-average responses. Decrease probability of below-average responses.**

```
Above average (A > 0) → r·A is positive → push probability up
Below average (A < 0) → r·A is negative → push probability down
```

⚠️ Same problem as PPO — if $r_i$ is very large, update becomes too big.
This is why clipping is needed.

---

## 🔹 5. Clipping Function

$$
\text{clip}(r_i(\theta),\ 1-\epsilon,\ 1+\epsilon)
$$

> **If the policy changed too much for this response — limit it.**

With $\epsilon = 0.2$, the allowed range is **[0.8, 1.2]**:

| $r_i$ value | After clip | What happened |
|-------------|------------|---------------|
| 1.1 | 1.1 | Inside range — no change |
| 1.5 | 1.2 | Too large — clipped down |
| 0.6 | 0.8 | Too small — clipped up |

Identical mechanism to PPO — just applied per response instead of per token.

---

## 🔹 6. Second Term — Clipped Objective

$$
\text{clip}(r_i(\theta),\ 1-\epsilon,\ 1+\epsilon) \cdot \hat{A}_i
$$

> Safe version of the update — ratio is bounded, updates stay controlled.

---

## 🔥 7. Why `min()` is Used ⭐

$$
\min\left(r_i(\theta) \cdot \hat{A}_i,\ \text{clip}(r_i(\theta), 1-\epsilon, 1+\epsilon) \cdot \hat{A}_i\right)
$$

Same logic as PPO — always picks the more conservative term.

### Case 1: Above Average Response $(\hat{A}_i > 0)$

```
r = 1.5   (policy increased probability a lot)
A = +1.34 (best response in group)

Unclipped = 1.5  × 1.34 = 2.01
Clipped   = 1.2  × 1.34 = 1.61

min(2.01, 1.61) = 1.61  ✅
```

> Increases probability of this response — but not as aggressively as raw gradient wanted.

### Case 2: Below Average Response $(\hat{A}_i < 0)$

```
r = 0.5    (policy decreased probability a lot)
A = -1.34  (worst response in group)

Unclipped = 0.5 × -1.34 = -0.67
Clipped   = 0.8 × -1.34 = -1.07

min(-0.67, -1.07) = -1.07  ✅
```

> Decreases probability of this response — but prevents over-correction.

---

## 🔹 8. KL Penalty Term

$$
\beta \cdot \mathbb{D}_{KL}\left[\pi_\theta \| \pi_{ref}\right]
$$

> **Prevents the policy from drifting too far from the reference model.**

```
If policy changes too much → KL increases → loss increases
→ acts as a brake on updates
```

| $\beta$ | Effect |
|---------|--------|
| Small (0.01) | More exploration — policy can change freely |
| Large (0.5) | Stays close to reference model |

Same purpose as PPO's KL penalty — just added directly to the loss instead of being a separate model.

---

## 💡 One Line Summary

```
PPO  → advantage = reward - V(s)        needs a value network
GRPO → advantage = (reward - group mean) / group std    just maths

Both use the same clipping mechanism
Both use KL penalty to prevent drift

GRPO replaces the value network
with the group itself as the baseline
```

In [6]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 9.4 MB/s eta 0:00:00a 0:00:01


In [9]:
import os, torch, trl
from transformers import AutoTokenizer
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

print("TRL version:", trl.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

model_name = "Qwen/Qwen1.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ── Dataset (raw text — GRPO handles tokenization internally) ──────────
dataset = Dataset.from_dict({
    "prompt": ["Explain gravity simply"] * 50
})

# ── Reward Function ────────────────────────────────────────────────────
def reward_fn(completions, **kwargs):
    rewards = []
    for c in completions:
        text  = c if isinstance(c, str) else c[0]["content"]
        score = 0.0
        if "force" in text.lower():        score += 1.0
        if 10 < len(text.split()) < 60:    score += 0.5
        rewards.append(score)
    return rewards

# ── GRPO Config ────────────────────────────────────────────────────────
config = GRPOConfig(
    output_dir                  = "./grpo_output",
    learning_rate               = 1e-5,
    per_device_train_batch_size = 1,
    num_train_epochs            = 1,
    
    max_completion_length       = 50,
    num_generations             = 2,
    generation_batch_size       = 2,   # ← must equal num_generations
    temperature                 = 0.7,
    beta                        = 0.05,
    epsilon                     = 0.2,
    report_to                   = "none",
)

# ── Trainer ────────────────────────────────────────────────────────────
trainer = GRPOTrainer(
    model            = model_name,
    args             = config,
    reward_funcs     = reward_fn,
    train_dataset    = dataset,
    processing_class = tokenizer,
)

# ── Train ──────────────────────────────────────────────────────────────
print("\n" + "━"*50)
print("  GRPO Training")
print("━"*50)
trainer.train()
# Save manually right after trainer.train()
trainer.save_model("./grpo_output")
print("\nGRPO complete ✓")

TRL version: 0.29.0
Device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GRPO Training
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Step,Training Loss
10,0.009386
20,0.008613
30,0.012457
40,0.011201
50,0.008732


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


GRPO complete ✓


In [13]:
# ── Load trained model and generate response ───────────────────────────
from transformers import AutoModelForCausalLM

# Load the trained model from output directory
trained_model = AutoModelForCausalLM.from_pretrained("./grpo_output")
trained_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
trained_model.to(device)

# ── Generate ───────────────────────────────────────────────────────────
prompt = "Explain gravity simply"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = trained_model.generate(
        **inputs,
        max_new_tokens   = 100,
        do_sample        = True,
        temperature      = 0.7,
        top_p            = 0.9,
        pad_token_id     = tokenizer.eos_token_id
    )

# ── Decode and print ───────────────────────────────────────────────────
response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],   # slice off prompt
    skip_special_tokens=True
)

print("Prompt  :", prompt)
print("Response:", response)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prompt  : Explain gravity simply
Response:  in a way that a five-year-old can understand. Gravity is a force that pulls objects towards a central point. It is the force that is responsible for the motion of the planets in our solar system. It is the force that we see when we look up at the sky. Gravity is a natural force that we observe, but it is not a force that we can understand.


In [ ]:
# ── Compare base model vs trained model ────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_model.eval()

with torch.no_grad():
    base_output = base_model.generate(
        **inputs,
        max_new_tokens = 100,
        do_sample      = True,
        temperature    = 0.7,
        top_p          = 0.9,
        pad_token_id   = tokenizer.eos_token_id
    )

base_response = tokenizer.decode(
    base_output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=" * 50)
print("Prompt         :", prompt)
print("=" * 50)
print("Base response  :", base_response)
print("-" * 50)
print("Trained response:", response)
print("=" * 50)